In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/Text Miners/Actual Work'
print('Connected:', BASE)


import os
print(os.listdir(BASE))


Mounted at /content/drive
Connected: /content/drive/MyDrive/Text Miners/Actual Work
['data', 'pre-processing', 'features', 'task1-sentiments', 'task2-topics', 'task3-combination', 'dashboard', 'BeforeYouStart.docx']


In [4]:
# !pip install scikit-learn==1.4 pandas numpy ftfy --quiet
!pip install vaderSentiment langdetect --quiet


In [5]:
import pandas as pd


df = pd.read_csv(f'{BASE}/data/MergedCleaned.csv')
print('Shape:', df.shape)
print(df.head(3))

Shape: (380505, 4)
   listing_id neighbourhood        room_type  \
0       27934   Ratchathewi  Entire home/apt   
1       27934   Ratchathewi  Entire home/apt   
2       27934   Ratchathewi  Entire home/apt   

                                            comments  
0  We stayed in the apartment for a week and we e...  
1  My girlfriend and I recently stayed in Nuttee'...  
2  I stayed for one month at the condo and was re...  


In [6]:
import sys
sys.path.append(f'{BASE}/pre-processing')
import preprocess
print('preprocess.py loaded')

preprocess.py loaded


In [7]:
# This adds two new columns to your dataframe:
#   cleaned_text — preprocessed review as a string
#   tokens       — preprocessed review as a list of words


df = preprocess.preprocess_dataframe(df)
print(df[['comments', 'cleaned_text', 'tokens']].head(3))
# Just clean and tokenise one review
cleaned, tokens = preprocess.clean_text('Great location!! Very clean place.')
print(cleaned)   # → 'great location very clean place'
print(tokens)    # → ['great', 'location', 'very', 'clean', 'place'

                                            comments  \
0  We stayed in the apartment for a week and we e...   
1  My girlfriend and I recently stayed in Nuttee'...   
2  I stayed for one month at the condo and was re...   

                                        cleaned_text  \
0  stayed apartment week enjoyed very much nuttee...   
1  girlfriend recently stayed nuttee condo month ...   
2  stayed one month condo realy pleased condo 19t...   

                                              tokens  
0  [stayed, apartment, week, enjoyed, very, much,...  
1  [girlfriend, recently, stayed, nuttee, condo, ...  
2  [stayed, one, month, condo, realy, pleased, co...  
great location very clean place
['great', 'location', 'very', 'clean', 'place']


In [8]:
import os
print(os.listdir(f'{BASE}/data'))

['raw_reviews.gsheet', 'EDA Portion', 'MergedCleaned.csv']


In [9]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

df['compound_score'] = df['comments'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

print("Done! Sample scores:")
print(df[['comments', 'compound_score']].head(3))

Done! Sample scores:
                                            comments  compound_score
0  We stayed in the apartment for a week and we e...          0.9729
1  My girlfriend and I recently stayed in Nuttee'...          0.9875
2  I stayed for one month at the condo and was re...          0.9913


In [20]:
# Check which threshold is better for checking sentiment analysis
for thresh in [0.05, 0.1, 0.2, 0.3]:
    pos = (df['compound_score'] >= thresh).sum()
    neg = (df['compound_score'] <= -thresh).sum()
    neu = ((df['compound_score'] > -thresh) & (df['compound_score'] < thresh)).sum()
    total = len(df)
    print(f"±{thresh}  →  pos: {pos/total*100:.1f}%  neg: {neg/total*100:.1f}%  neu: {neu/total*100:.1f}%")

# Sentiments for different thresholds
# ±0.05  →  pos: 96.0%  neg: 2.3%  neu: 1.8%
# ±0.1  →  pos: 95.8%  neg: 2.2%  neu: 2.0%
# ±0.2  →  pos: 95.4%  neg: 1.9%  neu: 2.7%
# ±0.3  →  pos: 94.5%  neg: 1.6%  neu: 3.9%

±0.05  →  pos: 96.0%  neg: 2.3%  neu: 1.8%
±0.1  →  pos: 95.8%  neg: 2.2%  neu: 2.0%
±0.2  →  pos: 95.4%  neg: 1.9%  neu: 2.7%
±0.3  →  pos: 94.5%  neg: 1.6%  neu: 3.9%


In [22]:
def assign_label(c):
    if c >= 0.30:    return 'positive'
    elif c <= -0.30: return 'negative'
    else:            return 'neutral'

df['label'] = df['compound_score'].apply(assign_label)

counts = df['label'].value_counts()
pcts   = df['label'].value_counts(normalize=True) * 100

for label in counts.index:
    print(f"{label:10s}  {counts[label]:>7,}  ({pcts[label]:.1f}%)")

# Overall sentiment for all the reviews
# positive    359,559  (94.5%)
# neutral      14,784  (3.9%)
# negative      6,162  (1.6%)

positive    359,559  (94.5%)
neutral      14,784  (3.9%)
negative      6,162  (1.6%)


In [19]:
# Apply final threshold ±0.3
df['label'] = df['compound_score'].apply(
    lambda c: 'positive' if c >= 0.3 else ('negative' if c <= -0.3 else 'neutral')
)

print("Final class balance:")
counts = df['label'].value_counts()
pcts   = df['label'].value_counts(normalize=True) * 100
for label in counts.index:
    print(f"{label:10s}  {counts[label]:>7,}  ({pcts[label]:.1f}%)")

# Export
import os
os.makedirs(f'{BASE}/data', exist_ok=True)
out = df[['listing_id', 'comments', 'compound_score', 'label']]
out.to_csv(f'{BASE}/data/labelled_reviews.csv', index=False)
print(f"\n✓ Saved {len(out):,} rows → {BASE}/data/labelled_reviews.csv")

# Overall class sentiments
# Final class balance:
# positive    359,559  (94.5%)
# neutral      14,784  (3.9%)
# negative      6,162  (1.6%)

Final class balance:
positive    359,559  (94.5%)
neutral      14,784  (3.9%)
negative      6,162  (1.6%)

✓ Saved 380,505 rows → /content/drive/MyDrive/Text Miners/Actual Work/data/labelled_reviews.csv
